In [10]:
import os, pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv('train.csv')
df = df[['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']]

X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=['Survived']), df['Survived'], test_size=0.2, random_state=42
)

# missing values bharna
si_age = SimpleImputer(strategy='mean')
si_embarked = SimpleImputer(strategy='most_frequent')
age = si_age.fit_transform(X_train[['Age']])
emb = si_embarked.fit_transform(X_train[['Embarked']])

# one hot encoding
ohe_sex = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_embarked = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
sex_enc = ohe_sex.fit_transform(X_train[['Sex']].values)
emb_enc = ohe_embarked.fit_transform(emb)

# sir wali order me features jodna
X_train_final = np.concatenate(
    (X_train[['Pclass', 'SibSp', 'Parch', 'Fare']].values, age, sex_enc, emb_enc), axis=1
)

# model train
clf = DecisionTreeClassifier()
clf.fit(X_train_final, y_train)

# models folder me save
os.makedirs('models', exist_ok=True)
pickle.dump(ohe_sex, open('models/ohe_sex.pkl', 'wb'))
pickle.dump(ohe_embarked, open('models/ohe_embarked.pkl', 'wb'))
pickle.dump(clf, open('models/clf.pkl', 'wb'))

X_train_final.shape

(712, 10)

In [8]:
import os
os.listdir()

['.ipynb_checkpoints',
 'clf.pkl',
 'models',
 'predict-using-pipeline.ipynb',
 'predict-without-pipeline.ipynb',
 'train.csv']

In [11]:
import pickle
import numpy as np

ohe_sex = pickle.load(open('models/ohe_sex.pkl', 'rb'))
ohe_embarked = pickle.load(open('models/ohe_embarked.pkl', 'rb'))
clf = pickle.load(open('models/clf.pkl', 'rb'))

# Pclass/gender/age/SibSp/Parch/Fare/Embarked
test_input = np.array([2, 'male', 31.0, 0, 0, 10.5, 'S'], dtype=object).reshape(1, 7)

test_input_sex = ohe_sex.transform(test_input[:, 1].reshape(1, 1))
test_input_embarked = ohe_embarked.transform(test_input[:, -1].reshape(1, 1))
test_input_age = test_input[:, 2].reshape(1, 1)

test_input_transformed = np.concatenate(
    (test_input[:, [0, 3, 4, 5]], test_input_age, test_input_sex, test_input_embarked), axis=1
)

print(test_input_transformed.shape)
clf.predict(test_input_transformed)

(1, 10)


array([0])

In [12]:
test_input = np.array([1, 'female', 28.0, 0, 0, 80.0, 'C'], dtype=object).reshape(1, 7)

sex = ohe_sex.transform(test_input[:, 1].reshape(1, 1))
emb = ohe_embarked.transform(test_input[:, -1].reshape(1, 1))
age = test_input[:, 2].reshape(1, 1)

final = np.concatenate((test_input[:, [0, 3, 4, 5]], age, sex, emb), axis=1)
clf.predict(final)

array([1])